In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sym
from sympy import pprint
np.set_printoptions(linewidth=180)
import time
# import scipy as sci
from scipy.sparse import csr_matrix as csr

def pp(M, rounding=3, imag_tolerance = 1e-12):
    if type(M) is not np.ndarray:
        print(M)
    else:
        M = M.real.round(rounding) + 1j*M.imag.round(rounding)
        if imag_tolerance is not None:
            M = np.where(np.abs(M.imag) < imag_tolerance, M.real, M)
        print(np.round(M, rounding))

def sparse_to_dense(size, list, inv = False):       # list format : [[],[],[]] ...locations of ones
    l = len(list)
    s = size
    adj = np.zeros([s,s])
    print(adj.shape)

    for i in list:
        adj[i[0], i[1]] = 1
        adj[i[1], i[0]] = 1

    if inv:
        adj = np.ones(s,s) - adj

    return adj


# sparse_to_dense(4, [[0,1], [1,2], [2,3], [3,0]])


### Results for 1 qubit each and 2 qubit - for the initial version of partial swap operator

#### this is the previous version the extended version that the professor asked for earlier this is NOT the same as the one in notes.

In [94]:
# Define the Hadamard gate matrix
H = (1/np.sqrt(2)) * np.array([
    [1, 1],
    [1, -1]
])

# Define the CNOT gate matrix
CNOT_a = np.array([
    [0, 1, 0, 0],
    [1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])
CNOT_b = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
])

CZ = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, -1]
])

manual_rot = np.array([
    [1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 1, 0, 0],
    [0, 0, 1, 0]
])





# Define the I (identity) matrix
I = np.eye(2)

# Define the I ⊗ H matrix
I_H = np.kron(I, H)
H_I = np.kron(H, I)

import numpy as np

def Rot(theta, pos = False):
    rot = np.array([
        [np.cos(theta / 2), -np.sin(theta / 2)],
        [np.sin(theta / 2), np.cos(theta / 2)]
    ])
    identity = np.eye(2)
    if pos == False:
        return np.kron(rot, identity)
    else: 
        return np.kron(identity, rot)


In [95]:
# result2 =  @ Rot_2 @ CNOT_a @Rot_2true
# result2 = Rot(-np.pi/4, 1) @ CNOT_a @ Rot(-np.pi/4, 1)
# result2 = CZ @ Rot(np.pi/4, 1) @ CZ @ Rot(-np.pi/4, 1)   # working

# np.transpose(manual_rot) @ CZ @ Rot(np.pi/4, 1) @ CZ @ Rot(-np.pi/4, 1) @ manual_rot # print individual matrices
print("\n \n CZ: \n ", CZ)
print("\n \n Rot(pi/4)_B: \n ", Rot(np.pi/4, 1))
print("\n \n Rot(-pi/4)_B: \n ", Rot(-np.pi/4, 1))
print("\n \n manual_rot: \n ", manual_rot)

result = np.transpose(manual_rot) @ CZ @ Rot(np.pi/4, 1) @ CZ @ Rot(-np.pi/4, 1) @ manual_rot

result = np.round(result, decimals=10)
print("\n \n Resulting matrix: \n ", result)



 
 CZ: 
  [[ 1  0  0  0]
 [ 0  1  0  0]
 [ 0  0  1  0]
 [ 0  0  0 -1]]

 
 Rot(pi/4)_B: 
  [[ 0.92387953 -0.38268343  0.         -0.        ]
 [ 0.38268343  0.92387953  0.          0.        ]
 [ 0.         -0.          0.92387953 -0.38268343]
 [ 0.          0.          0.38268343  0.92387953]]

 
 Rot(-pi/4)_B: 
  [[ 0.92387953  0.38268343  0.          0.        ]
 [-0.38268343  0.92387953 -0.          0.        ]
 [ 0.          0.          0.92387953  0.38268343]
 [-0.          0.         -0.38268343  0.92387953]]

 
 manual_rot: 
  [[1 0 0 0]
 [0 0 0 1]
 [0 1 0 0]
 [0 0 1 0]]

 
 Resulting matrix: 
  [[ 1.          0.          0.          0.        ]
 [ 0.          0.70710678  0.70710678  0.        ]
 [ 0.         -0.70710678  0.70710678  0.        ]
 [ 0.          0.          0.          1.        ]]


#### Generalizing to Two qubits 

In [96]:
# order of basis states: {b_00, b_01, b_02, b_03, b_10, b_11, b_12, b_13, b_20, b_21, b_22, b_23, b_30, b_31, b_32, b_33}
R2 = np.sqrt(2)         # √2, so that the matrix is neat looking
U2qubit_unnormalized = np.array([
    [1,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0],
    [0,  1,  0,  0,    1,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0],
    [0,  0,  1,  0,    0,  0,  0,  0,    1,  0,  0,  0,    0,  0,  0,  0],
    [0,  0,  0,  1,    0,  0,  0,  0,    0,  0,  0,  0,    1,  0,  0,  0],
    
    [0, -1,  0,  0,    1,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0],
    [0,  0,  0,  0,    0,  1,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0],
    [0,  0,  0,  0,    0,  0,  1,  0,    0,  1,  0,  0,    0,  0,  0,  0],
    [0,  0,  0,  0,    0,  0,  0,  1,    0,  0,  0,  0,    0,  1,  0,  0],
    
    [0,  0, -1,  0,    0,  0,  0,  0,    1,  0,  0,  0,    0,  0,  0,  0],
    [0,  0,  0,  0,    0,  0, -1,  0,    0,  1,  0,  0,    0,  0,  0,  0],
    [0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  1,  0,    0,  0,  0,  0],
    [0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  1,    0,  0,  1,  0],
    
    [0,  0,  0, -1,    0,  0,  0,  0,    0,  0,  0,  0,    1,  0,  0,  0],
    [0,  0,  0,  0,    0,  0,  0, -1,    0,  0,  0,  0,    0,  1,  0,  0],
    [0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0, -1,    0,  0,  1,  0],
    [0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  1]
])

U2qubit = (1/R2)* np.array([
    [R2,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0],
    [ 0,  1,  0,  0,    1,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0],
    [ 0,  0,  1,  0,    0,  0,  0,  0,    1,  0,  0,  0,    0,  0,  0,  0],
    [ 0,  0,  0,  1,    0,  0,  0,  0,    0,  0,  0,  0,    1,  0,  0,  0],
    
    [ 0, -1,  0,  0,    1,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0],
    [ 0,  0,  0,  0,    0, R2,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0],
    [ 0,  0,  0,  0,    0,  0,  1,  0,    0,  1,  0,  0,    0,  0,  0,  0],
    [ 0,  0,  0,  0,    0,  0,  0,  1,    0,  0,  0,  0,    0,  1,  0,  0],
    
    [ 0,  0, -1,  0,    0,  0,  0,  0,    1,  0,  0,  0,    0,  0,  0,  0],
    [ 0,  0,  0,  0,    0,  0, -1,  0,    0,  1,  0,  0,    0,  0,  0,  0],
    [ 0,  0,  0,  0,    0,  0,  0,  0,    0,  0, R2,  0,    0,  0,  0,  0],
    [ 0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  1,    0,  0,  1,  0],
    
    [ 0,  0,  0, -1,    0,  0,  0,  0,    0,  0,  0,  0,    1,  0,  0,  0],
    [ 0,  0,  0,  0,    0,  0,  0, -1,    0,  0,  0,  0,    0,  1,  0,  0],
    [ 0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0, -1,    0,  0,  1,  0],
    [ 0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0,  0,    0,  0,  0, R2]])

In [97]:
# experimenting with gates to reproduce the U2qubit matrix

In [98]:
# verifying U2qubit using sympy
# create sympy vector
a00, a01, a02, a03, a10, a11, a12, a13, a20, a21, a22, a23, a30, a31, a32, a33 = sym.symbols('a00 a01 a02 a03 a10 a11 a12 a13 a20 a21 a22 a23 a30 a31 a32 a33')
a_vec = sym.Matrix([a00, a01, a02, a03, a10, a11, a12, a13, a20, a21, a22, a23, a30, a31, a32, a33])
#apply the U2qubit matrix to the vector and print the result
result = R2* U2qubit @ a_vec
pprint(result)
# verified, correct



⎡1.4142135623731⋅a₀₀⎤
⎢                   ⎥
⎢ 1.0⋅a₀₁ + 1.0⋅a₁₀ ⎥
⎢                   ⎥
⎢ 1.0⋅a₀₂ + 1.0⋅a₂₀ ⎥
⎢                   ⎥
⎢ 1.0⋅a₀₃ + 1.0⋅a₃₀ ⎥
⎢                   ⎥
⎢  -a₀₁ + 1.0⋅a₁₀   ⎥
⎢                   ⎥
⎢1.4142135623731⋅a₁₁⎥
⎢                   ⎥
⎢ 1.0⋅a₁₂ + 1.0⋅a₂₁ ⎥
⎢                   ⎥
⎢ 1.0⋅a₁₃ + 1.0⋅a₃₁ ⎥
⎢                   ⎥
⎢  -a₀₂ + 1.0⋅a₂₀   ⎥
⎢                   ⎥
⎢  -a₁₂ + 1.0⋅a₂₁   ⎥
⎢                   ⎥
⎢1.4142135623731⋅a₂₂⎥
⎢                   ⎥
⎢ 1.0⋅a₂₃ + 1.0⋅a₃₂ ⎥
⎢                   ⎥
⎢  -a₀₃ + 1.0⋅a₃₀   ⎥
⎢                   ⎥
⎢  -a₁₃ + 1.0⋅a₃₁   ⎥
⎢                   ⎥
⎢  -a₂₃ + 1.0⋅a₃₂   ⎥
⎢                   ⎥
⎣1.4142135623731⋅a₃₃⎦


In [99]:

M = (np.kron(H_I@ I_H, H_I @ I_H)@np.kron(H_I@ I_H, H_I @ I_H))
#clean up the matrix
M = np.round(M, decimals=2)
print(np.linalg.norm(M-U2qubit))
M


2.6513085922847344


array([[ 1., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,
        -0., -0.,  0.],
       [-0.,  1.,  0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,
         0.,  0., -0.],
       [-0.,  0.,  1., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,
         0.,  0., -0.],
       [ 0., -0., -0.,  1., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,
        -0., -0.,  0.],
       [-0.,  0.,  0., -0.,  1., -0., -0.,  0.,  0., -0., -0.,  0., -0.,
         0.,  0., -0.],
       [ 0., -0., -0.,  0., -0.,  1.,  0., -0., -0.,  0.,  0., -0.,  0.,
        -0., -0.,  0.],
       [ 0., -0., -0.,  0., -0.,  0.,  1., -0., -0.,  0.,  0., -0.,  0.,
        -0., -0.,  0.],
       [-0.,  0.,  0., -0.,  0., -0., -0.,  1.,  0., -0., -0.,  0., -0.,
         0.,  0., -0.],
       [-0.,  0.,  0., -0.,  0., -0., -0.,  0.,  1., -0., -0.,  0., -0.,
         0.,  0., -0.],
       [ 0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  1.,  0., -0.,  0.,
        -0., -0.,  0.],
       [ 0., -0., -0.,  0., -0

### Creating General U Matrix for n,n qubits dimensions - mixes |0>|p> and |p>|0> and leaves all other states unchanged. 

In [100]:
#defining the U matrix for q qubits
def U_ab(q):    # (q+q) qubits is the total number of qubits in the system
    N = 2**q 

    def kron_delta(a, b):
        return 1 if a == b else 0
    
    def U_mn(m,n):
        i, j = divmod(m, N)
        k, l = divmod(n, N)

        if k == 0 and l == 0:
            return (kron_delta(i, 0) * kron_delta(j, 0))
        elif k == 0 and l != 0:
            return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
        elif k != 0 and l == 0:
            return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
        else:
            return (kron_delta(i, k) * kron_delta(j, l))

    # create matrix using for loops - numpy gives errors
    U = np.zeros((N**2, N**2))
    for m in range(N**2):
        for n in range(N**2):
            U[m, n] = U_mn(m, n)

    return U



In [101]:
# checking - working, unitary
U_1 = U_ab(2)
U_1_round = np.round(U_1, 3)
b = sym.Matrix(U_1_round)
pprint(b)                       # matrix is as expected

np.round((U_1 @ U_1.T),10)      # matrix is unitary

⎡1.0    0       0       0       0     0    0    0     0     0    0    0     0 
⎢                                                                             
⎢ 0   0.707     0       0     0.707   0    0    0     0     0    0    0     0 
⎢                                                                             
⎢ 0     0     0.707     0       0     0    0    0   0.707   0    0    0     0 
⎢                                                                             
⎢ 0     0       0     0.707     0     0    0    0     0     0    0    0   0.70
⎢                                                                             
⎢ 0   -0.707    0       0     0.707   0    0    0     0     0    0    0     0 
⎢                                                                             
⎢ 0     0       0       0       0    1.0   0    0     0     0    0    0     0 
⎢                                                                             
⎢ 0     0       0       0       0     0   1.0   0   

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

### Building the U_a (phase operator)

In [102]:
# defining the phase operator

def phase_operator(q, alpha = 'auto', analytic_prob = False):
    N = 2**q
    
    if q%2 == 0:
        alpha = 4*np.pi/q**2
    else:
        alpha = 4*np.pi/(q**2-1)

    if alpha != 'auto':
        alpha = alpha           # alpha is the coefficient in the Hamiltonian

    def qubit_sequence(a,q):
        b = np.binary_repr(a, width = q)       # string
        c = np.array([int(i) for i in b])   # array
        return c

    def edges_count(seq):   #seq is a binary sequence array
        p_0 = np.count_nonzero(seq == 0)
        p_1 = np.count_nonzero(seq == 1)
        return int(p_0*p_1)

    phases = []
    for i in range(N):
        cross_edges_i = edges_count(qubit_sequence(i, q)) 
        phase_i = np.exp(-1j*alpha*cross_edges_i)
        phases.append(phase_i)
    phases = np.array(phases)
    
    # create the phase operator
    P = np.diag(phases)

    if analytic_prob == True:                           # calculate the probability of measuring subsystem b in state |0>.
        prob_1_analytic = (.25/N)*np.sum(np.abs(phases-1)**2)
        return P, alpha, prob_1_analytic
    else:
        return P, alpha         
    # working, manually verified for q = 2, 3


#eg.
P, alpha = phase_operator(3)
pprint(sym.Matrix(np.round(P, 3)))          


⎡1.0   0     0     0     0     0     0     0 ⎤
⎢                                            ⎥
⎢ 0   -1.0   0     0     0     0     0     0 ⎥
⎢                                            ⎥
⎢ 0    0    -1.0   0     0     0     0     0 ⎥
⎢                                            ⎥
⎢ 0    0     0    -1.0   0     0     0     0 ⎥
⎢                                            ⎥
⎢ 0    0     0     0    -1.0   0     0     0 ⎥
⎢                                            ⎥
⎢ 0    0     0     0     0    -1.0   0     0 ⎥
⎢                                            ⎥
⎢ 0    0     0     0     0     0    -1.0   0 ⎥
⎢                                            ⎥
⎣ 0    0     0     0     0     0     0    1.0⎦


In [ ]:
# defining creating matrix states function

def matrix_basis(q):
    N = 2**q
    basis = []
    for i in range(N):
        state_i = np.zeros(N)
        state_i[i] = 1
        basis.append(state_i)
    return np.array(basis)

def superposition_state(q, coeffs = 'maximal_except_0'):        # careful, dont include the zeros
    N = 2**q
    if coeffs == 'maximal_except_0':
        coeffs = np.ones(N)
        coeffs[0] = 0

    basis = matrix_basis(q)
    
    c = np.array(coeffs).reshape(1, N)
    c = c/np.sqrt(np.sum(np.abs(c)**2))  # normalizing 
        
    if basis.shape[1] != c.shape[1]:
        raise ValueError('Error: Shapes of coefficients and basis do not match')
    if len(coeffs) != N:            # checking length
        raise ValueError('Error: The number of coefficients is not 2^q')

    sup_state = np.sum(c * basis, axis = 1, keepdims = True)
    if (np.linalg.norm(sup_state)-1) > 1e-10:
        raise ValueError('Error: The resultant state is not normalized')
    
    return sup_state



### Implementing the protocol on toy examples

#### For q = 2. We want the amplittudes of states |01> and |10> to be amplified.

In [104]:

U_ab_2 = U_ab(2)
Phi_2_a , alpha = phase_operator(2)
I_2_a = np.eye(4)
I_2_b = np.eye(4)
U_2 = U_ab_2 @ (np.kron(Phi_2_a, I_2_b)) @ U_ab_2      # full unitary matrix

Psi_2_a = superposition_state(2)      # homogeneous superposition state in 2 qubits or 4 dimensions
Zero_2_b= superposition_state(2, coeffs = [1, 0, 0, 0])  # |0> state for subsystem b
Psi_2a_0b = np.kron(Psi_2_a, Zero_2_b)

# To perform measurement, we have to find the probabilities of the corresponding collapse. Finding prob of subsystem b collapsing to |0> state, using operator I_2_a ⊗ |0><0|_b
# measurement operator
Proj_ab = np.kron(I_2_a, Zero_2_b @ Zero_2_b.conj().T)  # I@_a ⊗ |0><0|_b operator

In [125]:
# applying the first iteration
Psi_ab = U_2 @ Psi_2a_0b      
Psi_ab.T.round(4).real

# finding measurment probabilities for simulating the collapse later
# print(P_0_b.shape, Psi_ab.shape)

collapsed_Psi_ab = Proj_ab @ Psi_ab
prob_1 = np.sum(np.abs(collapsed_Psi_ab)**2)        # joint prob of Psi collapsing to |0> state in subsystem b (notation according to the notes)
prob_0 = 1 - prob_1

# Psi_2a_0b.T.round(4) , Psi_ab.T.round(4).real, collapsed_Psi_ab.T.round(4).real



In [126]:
# applying the second iteration
Psi_ab_2 = U_2 @ collapsed_Psi_ab
collapsed_Psi_ab_2 = Proj_ab @ Psi_ab_2

# pp(Psi_ab_2.T, 3)
pp(collapsed_Psi_ab_2.T, 3)

prob2 = np.sum(np.abs(collapsed_Psi_ab_2)**2)
prob2


[0  0  0  0  0.577  0  0  0  0.577  0  0  0  0  0  0  0]


0.6666666666666663

##### Discrepency in analytical prob and numerical one...a little below in the rough section are some checks. Everything checks out.


#### Rough: checking for errors for q =2

In [109]:
# printting all the matrices and states

# print("\n \n Phi_2_a: \n")
# pp(Phi_2_a, 2)
# print("\n \n P_0_b: \n")
# pp(P_0_b, 2)
print("\n \n Psi_2a_0b: \n")
pp(Psi_2a_0b.T, 2)

print("\n \n Psi_ab: \n")
pp(Psi_ab.T, 2)
# print("\n \n Psi_2a_0b: \n")
# pp(Psi_2a_0b.T, 2)
print("\n \n collapsed_Psi_ab: \n")
pp(collapsed_Psi_ab.T, 2)
#print the prob
print("\n \n prob_1: ", prob_1)


 
 Psi_2a_0b: 

[0  0  0  0  0.58  0  0  0  0.58  0  0  0  0.58  0  0  0]

 
 Psi_ab: 

[0  0  0  0.58  -0.58  0  0  0  -0.58  0  0  0  0  0  0  0]

 
 collapsed_Psi_ab: 

[0  0  0  0  -0.58  0  0  0  -0.58  0  0  0  0  0  0  0]

 
 prob_1:  0.6666666666666666


In [110]:
print("\n \n Phi: \n")
pp(Phi_2_a, 2)

print("\n \n phi kron I: \n")
pp(np.kron(Phi_2_a, I_2_b), 2)

# print("\n \n U_ab_2: \n")
# # pp(U_ab_2, 1)

# print("\n \n U_2: \n")
# # pp(U_2, 2)


 
 Phi: 

⎡1.0   0     0     0 ⎤
⎢                    ⎥
⎢ 0   -1.0   0     0 ⎥
⎢                    ⎥
⎢ 0    0    -1.0   0 ⎥
⎢                    ⎥
⎣ 0    0     0    1.0⎦

 
 phi kron I: 

⎡1.0   0    0    0    0     0     0     0     0     0     0     0     0    0  
⎢                                                                             
⎢ 0   1.0   0    0    0     0     0     0     0     0     0     0     0    0  
⎢                                                                             
⎢ 0    0   1.0   0    0     0     0     0     0     0     0     0     0    0  
⎢                                                                             
⎢ 0    0    0   1.0   0     0     0     0     0     0     0     0     0    0  
⎢                                                                             
⎢ 0    0    0    0   -1.0   0     0     0     0     0     0     0     0    0  
⎢                                                                             
⎢ 0    0    0    0  

In [111]:
# 

In [112]:
# checkig if matrices are unitarty

# pp((U_ab_2 @ U_ab_2.T).round(3), 3)

# pp((U_2 @ U_2.T), 4)

pp((Phi_2_a @ Phi_2_a.T).round(3), 3)

⎡1.0   0    0    0 ⎤
⎢                  ⎥
⎢ 0   1.0   0    0 ⎥
⎢                  ⎥
⎢ 0    0   1.0   0 ⎥
⎢                  ⎥
⎣ 0    0    0   1.0⎦


In [113]:
# writing the U_ab matrix manually as a list of lists, to later convert in numpy array, 16 by 16 sparse matrix.
R = np.sqrt(2)
Uab_manual_list = [[R, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0] 
, [0, -1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, R, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, R, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, R, 0, 0, 0, 0, 0, 0, 0, 0]
, [0, 0, -1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, R, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, R, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, R, 0, 0, 0, 0]
, [0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, R, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, R, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, R]]

In [114]:
Uab_manual = (1/R)*np.array(Uab_manual_list)

pp(Uab_manual, 2)

⎡1.0    0      0      0     0     0    0    0    0     0    0    0    0     0 
⎢                                                                             
⎢ 0   0.71     0      0    0.71   0    0    0    0     0    0    0    0     0 
⎢                                                                             
⎢ 0     0    0.71     0     0     0    0    0   0.71   0    0    0    0     0 
⎢                                                                             
⎢ 0     0      0    0.71    0     0    0    0    0     0    0    0   0.71   0 
⎢                                                                             
⎢ 0   -0.71    0      0    0.71   0    0    0    0     0    0    0    0     0 
⎢                                                                             
⎢ 0     0      0      0     0    1.0   0    0    0     0    0    0    0     0 
⎢                                                                             
⎢ 0     0      0      0     0     0   1.0   0    0  

In [115]:
# finding difference between the manual and the generated matrix
pp((Uab_manual - U_ab_2),5)  # should be zero

⎡0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎤
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                              ⎥
⎢0  0  0  0  0  0  0

#### Trying for q = 3

In [127]:
# trying for q =3 
U_ab_3 = U_ab(3)
Phi_3_a , alpha_3, prob_1_analytic = phase_operator(3, analytic_prob = True)
I_3_a = np.eye(8)
I_3_b = np.eye(8)
U_3 = U_ab_3 @ (np.kron(Phi_3_a, I_3_b)) @ U_ab_3      # full unitary matrix


In [128]:
#states
psi0 = superposition_state(3)
Zero0_b = superposition_state(3, coeffs = [1, 0, 0, 0, 0, 0, 0, 0])  # |0> state for subsystem b
Psi0a_0b = np.kron(psi0, Zero0_b)  # total 64 dimensions

#measurement operator
P0_b = np.kron(I_3_a, Zero0_b @ Zero0_b.conj().T)  # I@_a ⊗ |0><0|_b operator

In [131]:
# applying the first iteration
Psi_ab_3 = U_3 @ Psi0a_0b
collapsed_Psi_ab_3 = P0_b @ Psi_ab_3
prob_1_3 = np.sum(np.abs(collapsed_Psi_ab_3)**2)        

print(prob_1_3, prob_1_analytic)

#second iteration
Psi_ab_3_2 = U_3 @ collapsed_Psi_ab_3
collapsed_Psi_ab_3_2 = P0_b @ Psi_ab_3_2
prob_1_3_2 = np.sum(np.abs(collapsed_Psi_ab_3_2)**2)
pp(collapsed_Psi_ab_3_2.T, 3)

0.8571428571428565 0.75
[0  0  0  0  0  0  0  0  0.378  0  0  0  0  0  0  0  0.378  0  0  0  0  0  0  
0  0.378  0  0  0  0  0  0  0  0.378  0  0  0  0  0  0  0  0.378  0  0  0  0  
0  0  0  0.378  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]


### Converting above to class objects

In [7]:
class QuantumSystem:
    def __init__(self, q, alpha = 'auto'):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        self.alpha = alpha

        self.psi0_a = self.superposition_state()
        self.zero_b = self.superposition_state(coeffs = [1] + [0]*(self.N-1))
        self.psi_kron_0 = np.kron(self.psi0_a, self.zero_b)
        
        self.I_a = np.eye(self.N)
        self.I_b = np.eye(self.N)
        self.Proj_ab = np.kron(self.I_a, self.zero_b @ self.zero_b.conj().T)    # projection operator - Identity on a, projection on |0> on b
        
        self.Phase = np.diag(self.phases(alpha))                      # phase operator from the phases, "U_a" in notes
        self.Swap = self.U_ab()                                                 # swap operator, "U_ab" in notes

        self.U_big = self.Swap @ (np.kron(self.Phase, self.I_b)) @ self.Swap      # full unitary matrix
        
    def matrix_basis(self):
        q = self.q
        N = 2**q
        basis = []
        for i in range(N):
            state_i = np.zeros(N)
            state_i[i] = 1
            basis.append(state_i)
        return np.array(basis)

    def superposition_state(self, coeffs = 'maximal_except_0'):        # careful, dont include the zeros
        q = self.q
        N = self.N
        if coeffs == 'maximal_except_0':
            coeffs = np.ones(N)
            coeffs[0] = 0

        basis = self.matrix_basis()

        c = np.array(coeffs).reshape(1, N)
        c = c/np.sqrt(np.sum(np.abs(c)**2))  # normalizing 

        if basis.shape[1] != c.shape[1]:
            raise ValueError('Error: Shapes of coefficients and basis do not match')
        if len(coeffs) != N:            # checking length
            raise ValueError('Error: The number of coefficients is not 2^q')

        sup_state = np.sum(c * basis, axis = 1, keepdims = True)
        if (np.linalg.norm(sup_state)-1) > 1e-10:
            raise ValueError('Error: The resultant state is not normalized')
        return sup_state
    
    
    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U


    def phases(self, alpha = 'auto'):       # phases for the phase operator
        q, N = self.q, self.N
        # alpha is the coefficient in the Hamiltonian
        if q%2 == 0:            # creating alpha automatically, see notebook
            alpha = 4*np.pi/q**2
        else:
            alpha = 4*np.pi/(q**2-1)
        if alpha != 'auto':
            alpha = alpha           

        def qubit_sequence(a,q):                # to be used for creating a particular qubit sequence/permutation
            b = np.binary_repr(a, width = q)       # string     
            c = np.array([int(i) for i in b])   # array
            return c



        def edges_count(seq):   #seq is a binary sequence array
            p_0 = np.count_nonzero(seq == 0)
            p_1 = np.count_nonzero(seq == 1)
            return int(p_0*p_1)                 # for fully connected graph

        phases = []
        for i in range(N):
            cross_edges_i = edges_count(qubit_sequence(i, q))           # qubit sequence is creating a binary permutation (qubit states permutations) from the integer i from 0 to N-1.
            phase_i = np.exp(-1j*alpha*cross_edges_i)
            phases.append(phase_i)
        return np.array(phases)


    @staticmethod
    def norm_sq(state):        # returns sum of absolute squares ie. sum of probabilities
        return (np.abs(state)**2).sum()
    
    @staticmethod
    def normalize(state):
        if type(state) is not np.ndarray:
            raise ValueError('Error: State is not a numpy array')
        return (1/np.sqrt(QuantumSystem.norm_sq(state)))*state

        


#### For q = 2

In [4]:
# create system for q = 2
Q2 = QuantumSystem(2)

psi = Q2.psi_kron_0
U_big = Q2.U_big


# First Iteration
psi_prime = U_big @ psi  # operated state
# psi_prime.T.round(4)

psi_collapsed_unnorm = Q2.Proj_ab @ psi_prime
# psi_collapsed.T.round(4)

prob = np.sum(np.abs(psi_collapsed_unnorm)**2)        
prob_th = (1/4)*(1/(Q2.N-1))* (np.abs(Q2.phases()[1:] - 1)**2).sum()  # theoretical probability

print(prob, prob_th)

0.6666666666666666 0.6666666666666666


In [52]:
# Second Iteration
psi_prime_2 = U_big @ psi_collapsed_unnorm
psi_collapsed_2_unnorm = Q2.Proj_ab @ psi_prime_2
prob_2 = np.sum(np.abs(psi_collapsed_2_unnorm)**2)
prob_th_2 = (1/4)*(1/(Q2.N-1))*(1/prob_th) * (np.abs(Q2.phases()[1:] - 1)**4).sum()  # th prob of second iteration

pp(psi_prime_2.T)
pp(psi_collapsed_2_unnorm.T)
print(prob_2, prob_th_2)


# Checking  (uncomment)

# phases = Q2.phases()[1:]
# a = phases - 1
# b = np.abs(a)**4
# c = np.sum(b)

# phases, a, b, c


[[0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.577+0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.577+0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
  0.   +0.j 0.   +0.j]]
[[0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.577+0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.577+0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
  0.   +0.j 0.   +0.j]]
0.6666666666666663 4.0


#### (working for q = 2, except the prob_th_2) Trying the same for q=3.

In [60]:
Q3 = QuantumSystem(3)
psi = Q3.psi_kron_0
U_big = Q3.U_big

# First Iteration
psi_prime = U_big @ psi  # operated state
psi_collapsed_unnorm = Q3.Proj_ab @ psi_prime
prob = np.sum(np.abs(psi_collapsed_unnorm)**2)
prob_th = (1/4)*(1/(Q3.N-1))* (np.abs(Q3.phases()[1:] - 1)**2).sum()  # th prob, 1st iter.

# Second Iteration
psi_prime_2 = U_big @ psi_collapsed_unnorm
psi_collapsed_2_unnorm = Q3.Proj_ab @ psi_prime_2
prob_2 = np.sum(np.abs(psi_collapsed_2_unnorm)**2)
prob_th_2 = (1/4)*(1/(Q3.N-1))*(1/prob_th) * (np.abs(Q3.phases()[1:] - 1)**4).sum()  # th prob of second iteration

print(prob, prob_th, prob_2, prob_th_2)

# pp(psi.T)
# pp("\n")
# pp(psi_prime.T)
# pp("\n")
# pp(psi_collapsed.T)
# pp("\n")
# pp(psi_prime_2.T)
# pp("\n")
# pp(psi_collapsed_2.T)
# pp("\n")

# print(prob, prob_th, prob_2, prob_th_2)

0.8571428571428565 0.8571428571428571 0.857142857142856 4.0


#### Now trying for q = 4,5,6

In [88]:
# doing same for two iteration for q = 4
Q4 = QuantumSystem(4)
psi = Q4.psi_kron_0
U_big = Q4.U_big

# First Iteration

psi_prime = U_big @ psi  # operated state
psi_collapsed_unnorm = Q4.Proj_ab @ psi_prime
prob = np.sum(np.abs(psi_collapsed_unnorm)**2)
prob_th = (1/4)*(1/(Q4.N-1))* (np.abs(Q4.phases()[1:] - 1)**2).sum()  # th prob, 1st iter.

# Second Iteration
psi_collapsed = Q4.normalize(psi_collapsed_unnorm)

psi_prime_2 = U_big @ psi_collapsed_unnorm
psi_collapsed_2_unnorm = Q4.Proj_ab @ psi_prime_2
prob_2 = Q4.norm_sq(psi_collapsed_2_unnorm)
prob_th_2 = (1/16)*(1/(Q4.N-1))*(1/prob_th) * (np.abs(Q4.phases()[1:] - 1)**4).sum()  # th prob of second iteration

# print(prob_2, prob_th_2)
print(prob, prob_2)

Q4.phases().round(3)

0.8552284749830787 0.7885618083164119


array([ 1.   +0.j   , -0.707-0.707j, -0.707-0.707j, -1.   -0.j   , -0.707-0.707j, -1.   -0.j   , -1.   -0.j   , -0.707-0.707j, -0.707-0.707j,
       -1.   -0.j   , -1.   -0.j   , -0.707-0.707j, -1.   -0.j   , -0.707-0.707j, -0.707-0.707j,  1.   +0.j   ])

In [21]:
# third iteration
psi_collapsed_2 = Q4.normalize(psi_collapsed_2_unnorm)
psi_prime_3 = U_big @ psi_collapsed_2
psi_collapsed_3_unnorm = Q4.Proj_ab @ psi_prime_3
prob_3 = Q4.norm_sq(psi_collapsed_3_unnorm)
# prob_th_3 = (1/4)*(1/(Q4.N-1))*(1/prob)*(1/prob_2) * (np.abs(Q4.phases()[1:] - 1)**6).sum()  # guess - th prob of third iteration

print('','prob3 ', prob_3)

# fourth iteration
psi_collapsed_3 = Q4.normalize(psi_collapsed_3_unnorm)
psi_prime_4 = U_big @ psi_collapsed_3
psi_collapsed_4_unnorm = Q4.Proj_ab @ psi_prime_4

prob_4 = Q4.norm_sq(psi_collapsed_4_unnorm)
print('','prob4 ', prob_4)

# fifth iteration
psi_collapsed_4 = Q4.normalize(psi_collapsed_4_unnorm)
psi_prime_5 = U_big @ psi_collapsed_4
psi_collapsed_5_unnorm = Q4.Proj_ab @ psi_prime_5

prob_5 = Q4.norm_sq(psi_collapsed_5_unnorm)
print('','prob5 ', prob_5) 

# sixth iteration
psi_collapsed_5 = Q4.normalize(psi_collapsed_5_unnorm)
psi_prime_6 = U_big @ psi_collapsed_5
psi_collapsed_6_unnorm = Q4.Proj_ab @ psi_prime_6

prob_6 = Q4.norm_sq(psi_collapsed_6_unnorm)
print('','prob6 ', prob_6)


 prob3  0.9278388088634739
 prob4  0.9336162393990847
 prob5  0.9393090206124595
 prob6  0.9448498948717806


Works great, Probs are succesively increasing!

For q = 4, - the solution of the max cut problem is split the 4 vertices into half each and that would correspond to the maximum number of cross edges which would be 4. So, we want one of the non-zero coefficients in the state and resultant of a few iterations to correspond to a basis state which is permutation of state 0011.
Checking that below: 

In [89]:
psi_collapsed_6 = Q4.normalize(psi_collapsed_6_unnorm)
# pp(psi_collapsed_6.T, 5)
# pp((np.abs(psi_collapsed_6.T)**2).real.astype(np.float64), 3)

cc = np.abs(psi_collapsed_6.T)**2
pp(psi_collapsed_6.T, 2)

mask = cc > .05

flat_ind = np.nonzero(mask)
pp(flat_ind)

print('ind =', np.nonzero(mask.flatten()))


qubit_binaries = np.vectorize(lambda x: format(x, '08b'))
dd = qubit_binaries(flat_ind[1])
dd


[[ 0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j
   0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j   -0.15+0.15j  0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j
   0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j   -0.15+0.15j  0.  +0.j    0.  +0.j    0.  +0.j
   0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j
   0.33+0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j
   0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j   -0.15+0.15j  0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j
   0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.  +0.j    0.33+0.j    0.  +0.j    0.  +0.j    0

array(['00110000', '01010000', '01100000', '10010000', '10100000', '11000000'], dtype='<U8')

Great! the above binaries are simply qubits in their 0 or 1 states the first 4 are the system qubits, the last 4 are the ancilla qubits and what we have just found is we filtered the basis states with the highest probability from the result resultant state after the 6 iterations and then we found their indices and we converted them to binary representation to represent the qubit states so in all of them we see that the first 4 qubits which are the system qubits always have 2 of them 0 and 2 of them 1 indicating that these are the states which are solution to the max cut problem.

In [5]:
# for q = 5 similar code

Q5 = QuantumSystem(5)
psi = Q5.psi_kron_0
U_big = Q5.U_big

# First Iteration

psi_prime = U_big @ psi  # operated state
psi_collapsed_unnorm = Q5.Proj_ab @ psi_prime
prob = np.sum(np.abs(psi_collapsed_unnorm)**2)
prob_th = (1/4)*(1/(Q5.N-1))* (np.abs(Q5.phases()[1:] - 1)**2).sum()  # th prob, 1st iter.

# Second Iteration
psi_collapsed = Q5.normalize(psi_collapsed_unnorm)

psi_prime_2 = U_big @ psi_collapsed_unnorm
psi_collapsed_2_unnorm = Q5.Proj_ab @ psi_prime_2
prob_2 = Q5.norm_sq(psi_collapsed_2_unnorm)


# third iteration
psi_collapsed_2 = Q5.normalize(psi_collapsed_2_unnorm)
psi_prime_3 = U_big @ psi_collapsed_2
psi_collapsed_3_unnorm = Q5.Proj_ab @ psi_prime_3

prob_3 = Q5.norm_sq(psi_collapsed_3_unnorm)

# fourth iteration
psi_collapsed_3 = Q5.normalize(psi_collapsed_3_unnorm)
psi_prime_4 = U_big @ psi_collapsed_3
psi_collapsed_4_unnorm = Q5.Proj_ab @ psi_prime_4

prob_4 = Q5.norm_sq(psi_collapsed_4_unnorm)

# fifth iteration
psi_collapsed_4 = Q5.normalize(psi_collapsed_4_unnorm)
psi_prime_5 = U_big @ psi_collapsed_4
psi_collapsed_5_unnorm = Q5.Proj_ab @ psi_prime_5

prob_5 = Q5.norm_sq(psi_collapsed_5_unnorm)

# sixth iteration
psi_collapsed_5 = Q5.normalize(psi_collapsed_5_unnorm)
psi_prime_6 = U_big @ psi_collapsed_5
psi_collapsed_6_unnorm = Q5.Proj_ab @ psi_prime_6

prob_6 = Q5.norm_sq(psi_collapsed_6_unnorm)

print(prob, prob_2, prob_3, prob_4, prob_5, prob_6)

0.8870967741935483 0.826612903225806 0.9451219512195115 0.9564516129032258 0.9658516020236079 0.9734831951113049


In [6]:
psi_collapsed_6 = Q5.normalize(psi_collapsed_6_unnorm)

cc = np.abs(psi_collapsed_6.T)**2

# pp(cc[0][512:])

mask = cc > .04

flat_ind = np.nonzero(mask)
qubit_binaries = np.vectorize(lambda x: format(x, '010b'))

dd = qubit_binaries(flat_ind[1])    
dd


array(['0001100000', '0010100000', '0011000000', '0011100000', '0100100000', '0101000000', '0101100000', '0110000000', '0110100000', '0111000000',
       '1000100000', '1001000000', '1001100000', '1010000000', '1010100000', '1011000000', '1100000000', '1100100000', '1101000000', '1110000000'],
      dtype='<U10')

### Rough - manual given graphs

In [1]:
import numpy as np

qubit_string = '101'
permut_arr = np.array([int(i) for i in qubit_string])

permut_arr[permut_arr == 0] = -1

col_indices, row_indices = np.meshgrid(permut_arr, permut_arr)  # eg. for permutation 101, col_indices will have columns 1,0 and 1; row_indices will have rows 1,0 and 1. Multiplying these will give only the cross edges to be 0, remaining will be 1. 

CE_raw = col_indices * row_indices
CE_raw = np.triu(CE_raw, k = 1)        # upper triangular part of the matrix to avoid double counting
CE_raw[CE_raw == 1] = 0             # inverting, to change cross edges from 0 to -1 (and latter to 1)
cross_edge_arr = (-1)*CE_raw                       # changing cross edges from -1 to 1

cross_edges_count = np.sum(cross_edge_arr)

cross_edge_arr ,cross_edges_count



(array([[0, 1, 0],
        [0, 0, 1],
        [0, 0, 0]]),
 2)

In [2]:
# graph will be given, consisting of positive integer q (the # of qubits) and the adjacency martix of the graph as numpy array. Graph = (q, adj_matrix)

graph_sq = (4, np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]]))  # 4 qubits, square graph

q, adj_matrix = graph_sq

Q4_sq = QuantumSystem(q)



array([[0, 1, 0, 1],
       [1, 0, 1, 0],
       [0, 1, 0, 1],
       [1, 0, 1, 0]])

In [58]:
def qubit_sequence(x,q):
    b = np.binary_repr(x, width = q)       # binary string for the integer/basis state x (out of 2**q states) 
    c = np.array([int(i) for i in b])       # eg. '101' -> [1, 0, 1]
    return c

# permut_arr is the array for the basis state x (which is a permutation of q qubits)

def cross_edges_max_arr(permut_arr):             #all possible cross edges for this permutation (will add graph adjacency matrix later)
    permut_arr[permut_arr == 0] = -1            # calling the zeroes as -1, for distinguishing the cross edges upon multiplication
    col_indices, row_indices = np.meshgrid(permut_arr, permut_arr)  # eg. for permutation 101, col_indices will have columns 1,-1 and 1; row_indices will have rows 1,-1 and 1.

    CE_raw = col_indices * row_indices      # cross edges are -1, rest are 1
    CE_raw = np.triu(CE_raw, k = 1)        # upper triangular part of the matrix to avoid double counting
    CE_raw[CE_raw == 1] = 0             # inverting, to change cross edges from 0 to -1 (and latter to 1)
    cross_edge_arr = (-1)*CE_raw                       # changing cross edges from -1 to 1
    return cross_edge_arr                    # returns upper triangular array of 1 and 0s, 1 for cross edge, 0 for self edge or no edge


def phi_x(x, q, adj):
    """ Returns phi and alpha for the particular basis state/partition x, given the adjacency matrix of the graph."""
    partition_string = qubit_sequence(x, q)
    partition_string_copy = partition_string.copy()

    C_E_max_arr_x = cross_edges_max_arr(partition_string_copy)
    total_edges = np.sum(adj) / 2  # adj is the adjacency matrix of the given graph
    C_E_actual = C_E_max_arr_x * adj
    C_E_count = np.sum(C_E_actual)
    alpha = -np.pi / (2 * total_edges)
    phi = -2 * alpha * C_E_count
    return phi


# defining the phase operator
def phase_diag(q, adj_matrix):
    N = 2**q
    E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph 
    alpha = -np.pi/(2*E_t)          

    phase_list = []
    for i in range(N):
        phase_list.append(phi_x(i, q, adj_matrix))
    
    return phase_list, alpha

In [ ]:
adj_matrix = np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]])
q = 4

phase_diag(q, adj_matrix)[2]

In [47]:
# trying running phi function for square graph

adj_sq = np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]])

# pp(np.binary_repr(3, width = 4))
phi_x(5, 4, adj_sq)


Partition string: [0 1 0 1]
Cross edges max array: [[0 1 0 1]
 [0 0 1 0]
 [0 0 0 1]
 [0 0 0 0]]
Total edges: 4.0
Cross edges actual: [[0 1 0 1]
 [0 0 1 0]
 [0 0 0 1]
 [0 0 0 0]]
Cross edges count: 4
Alpha (auto-calculated): -0.39269908169872414
Phi_x: 3.141592653589793


(array([0, 1, 0, 1]), 3.141592653589793, -0.39269908169872414)

In [39]:
def string_to_array(string):        
    return np.array([int(i) for i in string])

def string_to_array_vectorized(string_array):
    return np.vectorize(string_to_array)(string_array)

string_to_array_vectorized(xx)



ValueError: setting an array element with a sequence.

### For General graph (not necesarilly complete)

In [8]:
# defining class for the general graph as q+q qubit quantum system.
# to take in adjacency matrix, number of qubits. Will have methods to calculate the phase operator, the swap operator, the full unitary matrix, the superposition state, the measurement operator etc.
#
class QuantumSystem:
    def __init__(self, q, adj_matrix):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        self.basis_states = np.array([np.eye(self.N)[i] for i in range(self.N)])        # basis states for the q-qubits, list of N = 2^q states
        
        self.adj_matrix = adj_matrix
        self.E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph
        self.alpha = -np.pi/(2*self.E_t)

        self.psi0 = self.superposition_state()                              # homogeneous superposition state in q system qubits, 2^q dim.
        self.zero = self.superposition_state(coeffs = [1] + [0]*(self.N-1))  # |0> state for q anicillas
        self.psi0_tensor_zero = np.kron(self.psi0, self.zero)                  # to be sent in, 2^(2q) dim.
        
        self.I_a, self.I_b = np.eye(self.N), np.eye(self.N)                     
        self.Proj_ab = np.kron(self.I_a, self.zero @ self.zero.conj().T)    # projection operator - Identity on the system, projection on |0> state of ancillas
    
        self.phase_list = self.phases(self.q, self.adj_matrix)      # list of phases ANGLES
    
        self.Phase = np.diag(np.exp(1j*self.phase_list))                      # phase operator from the phases, "U_a" in notes
        self.Swap = self.U_ab()                                                 # swap operator, "U_ab" in notes
        self.U_big = self.Swap @ (np.kron(self.Phase, self.I_b)) @ self.Swap      # full unitary matrix

    # instead of defining a method for creating matrix basis we will do it as a class variable
    
    def superposition_state(self, coeffs = 'maximal_except_0'):        # careful, dont include the zeros
        q = self.q
        N = self.N
        if coeffs == 'maximal_except_0':
            coeffs = np.ones(N)
            coeffs[0] = 0

        c = np.array(coeffs).reshape(1, N)
        c = c/np.sqrt(np.sum(np.abs(c)**2))

        if self.basis_states.shape[1] != c.shape[1]:
            raise ValueError('Error: Shapes of coefficients and basis do not match')
        if len(coeffs) != N:            # checking length
            raise ValueError('Error: The number of coefficients is not 2^q')
        
        sup_state = np.sum(c * self.basis_states, axis = 1, keepdims = True)
        if (np.linalg.norm(sup_state)-1) > 1e-10:
            raise ValueError('Error: The resultant state is not normalized')
        return sup_state

    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U

    # defining the phase operator after defining the phi_x function for an phase of an individual permutation/basis state

    @staticmethod
    def qubit_sequence(x, q):
        b = np.binary_repr(x, width = q)       # binary string for the integer/basis state x (out of 2**q states) 
        c = np.array([int(i) for i in b])       # eg. '101' -> [1, 0, 1]
        return c

    # permut_arr is the array for the basis state x (which is a permutation of q qubits)

    @staticmethod
    def cross_edges_max_arr(permut_arr):             #all possible cross edges for this permutation (will add graph adjacency matrix later)
        permut_arr[permut_arr == 0] = -1            # calling the zeroes as -1, for distinguishing the cross edges upon multiplication
        col_indices, row_indices = np.meshgrid(permut_arr, permut_arr)  # eg. for permutation 101, col_indices will have columns 1,-1 and 1; row_indices will have rows 1,-1 and 1.

        CE_raw = col_indices * row_indices      # cross edges are -1, rest are 1
        CE_raw = np.triu(CE_raw, k = 1)        # upper triangular part of the matrix to avoid double counting
        CE_raw[CE_raw == 1] = 0             # inverting, to change cross edges from 0 to -1 (and latter to 1)
        cross_edge_arr = (-1)*CE_raw                       # changing cross edges from -1 to 1
        return cross_edge_arr                    # returns upper triangular array of 1 and 0s, 1 for cross edge, 0 for self edge or no edge

    def phi_x(self, x, q, adj):                 # returns the angle phi for the basis state/partition x
        """ Returns phi and alpha for the particular basis state/partition x, given the adjacency matrix of the graph."""
        partition_string = self.qubit_sequence(x, q)
        partition_string_copy = partition_string.copy()

        C_E_max_arr_x = self.cross_edges_max_arr(partition_string_copy)
        total_edges = np.sum(adj) / 2  # adj is the adjacency matrix of the given graph
        C_E_actual = C_E_max_arr_x * adj
        C_E_count = np.sum(C_E_actual)
        alpha = -np.pi / (2 * total_edges)
        phi = -2 * alpha * C_E_count
        return phi

    def phases(self, q, adj_matrix):        # runs phi_x for all basis states/partitions; returns the list of PHASES (angles, NOT exponential of them) for the whole basis
        N = 2**q
        E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph 
        phase_list = []
        for i in range(N):
            phase_list.append(self.phi_x(i, q, adj_matrix))
        phases = np.array(phase_list)
        return phases

    @staticmethod
    def norm_sq(state):        # returns sum of absolute squares ie. sum of probabilities
        return (np.abs(state)**2).sum()
    
    @staticmethod
    def normalize(state):
        if type(state) is not np.ndarray:
            raise ValueError('Error: State is not a numpy array')
        return (1/np.sqrt(QuantumSystem.norm_sq(state)))*state
    
    # @staticmethod
    # def tensor_div(arr,b):            # arr is tenosr prod of two states, b is the second state with all zeroes except one 1 at b'th position
    #     arr_copy = arr.copy()
    # @staticmethod
    # def state_to_qubit:


In [117]:
# streamlining, reducing RAM usage

import numpy as np
class QuantumSystem:
    def __init__(self, q, adj_matrix):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        
        self.adj_matrix = adj_matrix
        self.E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph
        self.alpha = -np.pi/(2*self.E_t)
        
        #Cached variables
        self.cached_basis_states = None
        self.cached_psi0 = None
        self.cached_zero = None
        self.cached_psi0_tensor_zero = None
        self.cached_I_a =  None
        self.cached_Proj_ab = None
        self.cached_phase_list = None
        # self.cached_Phase = None
        # self.cached_Swap = None
        self.cached_U_big = None

    # instead of defining a method for creating matrix basis we will do it as a class variable
    @property
    def Identity_N(self):
        if self.cached_basis_states is None:
            self.cached_basis_states = np.eye(self.N)
        return self.cached_basis_states         # returns identity matrix of size 2^q = N
        
    @property
    def psi0(self):
        if self.cached_psi0 is None:
            self.cached_psi0 = (1/np.sqrt(self.N-1))*np.sum(self.Identity_N[1:], axis = 0, keepdims = True).T
        return self.cached_psi0
    
    @property
    def zero(self):
        if self.cached_zero is None:
            self.cached_zero = np.eye(self.N)[0].reshape(1, self.N).T
        return self.cached_zero

    @property
    def psi0_tensor_zero(self):
        if self.cached_psi0_tensor_zero is None:
            self.cached_psi0_tensor_zero = np.kron(self.psi0, self.zero)
        return self.cached_psi0_tensor_zero

    @property
    def Proj_ab(self):
        if self.cached_Proj_ab is None:
            self.cached_Proj_ab = np.kron(self.Identity_N, self.zero @ self.zero.conj().T)  # projection operator
        return self.cached_Proj_ab

    @property
    def phase_list(self):
        if self.cached_phase_list is None:
            self.cached_phase_list = self.phases(self.q, self.adj_matrix)
        return self.cached_phase_list           # list of phases ANGLES,  NOT the exponentials of them

    @property
    def Phase(self):
        return np.diag(np.exp(1j*self.phase_list))

    @property
    def Swap(self):
        return self.U_ab()
    
    @property
    def U_big(self):
        if self.cached_U_big is None:
            self.cached_U_big = self.Swap @ (np.kron(self.Phase, self.Identity_N)) @ self.Swap
        return self.cached_U_big

    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U

    # defining the phase operator after defining the phi_x function for an phase of an individual permutation/basis state

    @staticmethod
    def qubit_sequence(x, q):
        b = np.binary_repr(x, width = q)       # binary string for the integer/basis state x (out of 2**q states) 
        c = np.array([int(i) for i in b])       # eg. '101' -> [1, 0, 1]
        return c

    # permut_arr is the array for the basis state x (which is a permutation of q qubits)

    @staticmethod
    def cross_edges_max_arr(permut_arr):             #all possible cross edges for this permutation (will add graph adjacency matrix later)
        permut_arr[permut_arr == 0] = -1            # calling the zeroes as -1, for distinguishing the cross edges upon multiplication
        col_indices, row_indices = np.meshgrid(permut_arr, permut_arr)  # eg. for permutation 101, col_indices will have columns 1,-1 and 1; row_indices will have rows 1,-1 and 1.

        CE_raw = col_indices * row_indices      # cross edges are -1, rest are 1
        CE_raw = np.triu(CE_raw, k = 1)        # upper triangular part of the matrix to avoid double counting
        CE_raw[CE_raw == 1] = 0             # inverting, to change cross edges from 0 to -1 (and latter to 1)
        cross_edge_arr = (-1)*CE_raw                       # changing cross edges from -1 to 1
        return cross_edge_arr                    # returns upper triangular array of 1 and 0s, 1 for cross edge, 0 for self edge or no edge

    def phi_x(self, x, q, adj):                 # returns the angle phi for the basis state/partition x
        """ Returns phi and alpha for the particular basis state/partition x, given the adjacency matrix of the graph."""
        partition_string = self.qubit_sequence(x, q)
        partition_string_copy = partition_string.copy()

        C_E_max_arr_x = self.cross_edges_max_arr(partition_string_copy)
        total_edges = np.sum(adj) / 2  # adj is the adjacency matrix of the given graph
        C_E_actual = C_E_max_arr_x * adj
        C_E_count = np.sum(C_E_actual)
        alpha = -np.pi / (2 * total_edges)
        phi = -2 * alpha * C_E_count
        return phi

    def phases(self, q, adj_matrix):        # runs phi_x for all basis states/partitions; returns the list of PHASES (angles, NOT exponential of them) for the whole basis
        N = 2**q
        E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph 
        phase_list = []
        for i in range(N):
            phase_list.append(self.phi_x(i, q, adj_matrix))
        phases = np.array(phase_list)
        return phases

    @staticmethod
    def norm_sq(state):        # returns sum of absolute squares ie. sum of probabilities
        return (np.abs(state)**2).sum()
    
    @staticmethod
    def normalize(state):           # normalizes the state
        if type(state) is not np.ndarray:
            raise ValueError('Error: State is not a numpy array')
        return (1/np.sqrt(QuantumSystem.norm_sq(state)))*state
    
    # @staticmethod
    # def tensor_div(arr,b):            # arr is tenosr prod of two states, b is the second state with all zeroes except one 1 at b'th position
    #     arr_copy = arr.copy()
    # @staticmethod
    # def state_to_qubit:


In [1]:
# circumveniting Swap matrices by using analytical expressions

import numpy as np
# streamlining, reducing RAM usage

import numpy as np
class QuantumSystem:
    def __init__(self, q, adj_matrix):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        
        self.adj_matrix = adj_matrix
        self.E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph
        self.alpha = -np.pi/(2*self.E_t)
        
        #Cached variables
        self.cached_basis_states = None
        self.cached_psi0 = None
        self.cached_zero = None
        self.cached_psi0_tensor_zero = None
        self.cached_I_a =  None
        self.cached_Proj_ab = None
        self.cached_phase_list = None
        # self.cached_Phase = None
        # self.cached_Swap = None
        self.cached_U_big = None
        
        self.cached_prob1 = None
        self.cached_prob2 = None
        self.cached_psi_iter2 = None
        

    # instead of defining a method for creating matrix basis we will do it as a class variable
    @property
    def Identity_N(self):
        if self.cached_basis_states is None:
            self.cached_basis_states = np.eye(self.N)
        return self.cached_basis_states         # returns identity matrix of size 2^q = N
        
    @property
    def psi0(self):
        if self.cached_psi0 is None:
            self.cached_psi0 = (1/np.sqrt(self.N-1))*np.sum(self.Identity_N[1:], axis = 0, keepdims = True).T
        return self.cached_psi0
    
    @property
    def zero(self):
        if self.cached_zero is None:
            self.cached_zero = np.eye(self.N)[0].reshape(1, self.N).T
        return self.cached_zero

    @property
    def psi0_tensor_zero(self):
        if self.cached_psi0_tensor_zero is None:
            self.cached_psi0_tensor_zero = np.kron(self.psi0, self.zero)
        return self.cached_psi0_tensor_zero

    @property
    def Proj_ab(self):
        if self.cached_Proj_ab is None:
            self.cached_Proj_ab = np.kron(self.Identity_N, self.zero @ self.zero.conj().T)  # projection operator
        return self.cached_Proj_ab

    @property
    def phase_angles(self):
        if self.cached_phase_list is None:
            self.cached_phase_list = self.phases(self.q, self.adj_matrix).reshape(self.N, 1)     
        return self.cached_phase_list           # COLUMN of phases ANGLES

    @property
    def phase_list(self):       # column of phases (not angles)
        return np.exp(1j*self.phase_angles)
    
    
    @property
    def Phase(self):        # phase operator
        return np.diag(np.exp(1j*self.phase_angles.reshape(-1)))

    @property
    def p1(self):
        if self.cached_prob1 is None:
            self.cached_prob1 = (.25/(self.N-1))*((np.abs(self.phase_list[1:] - 1)**2).sum())
        return self.cached_prob1

    @property
    def p2(self):
        if self.cached_prob2 is None:
            self.cached_prob2 = (.25/((self.N-1)*self.p1))*((np.abs(self.phase_list[1:] - 1)**4).sum())
        return self.cached_prob2
    
    @property
    def psi_iter2(self):
        if self.cached_psi_iter2 is None:
            norm_coeff = 1/np.sqrt(4*(self.N -1)*self.p1*self.p2)
            psi_iter_2_a = norm_coeff*(((((self.phase_list[1:] - 1))**2)*self.Identity_N[1:]).sum(axis = 0, keepdims = True)).T
            self.cached_psi_iter2 = np.kron(psi_iter_2_a, self.zero)                        # projection/collapse on ancilla's zero state
        return self.cached_psi_iter2

    @property
    def Swap(self):
        return self.U_ab()  
    
    @property
    def U_big(self):
        if self.cached_U_big is None:
            self.cached_U_big = self.Swap @ (np.kron(self.Phase, self.Identity_N)) @ self.Swap
        return self.cached_U_big

    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U

    # defining the phase operator after defining the phi_x function for an phase of an individual permutation/basis state

    @staticmethod
    def qubit_sequence(x, q):
        b = np.binary_repr(x, width = q)       # binary string for the integer/basis state x (out of 2**q states) 
        c = np.array([int(i) for i in b])       # eg. '101' -> [1, 0, 1]
        return c
    # permut_arr is the array for the basis state x (which is a permutation of q qubits)
    @staticmethod
    def cross_edges_max_arr(permut_arr):             #all possible cross edges for this permutation (will add graph adjacency matrix later)
        permut_arr[permut_arr == 0] = -1            # calling the zeroes as -1, for distinguishing the cross edges upon multiplication
        col_indices, row_indices = np.meshgrid(permut_arr, permut_arr)  # eg. for permutation 101, col_indices will have columns 1,-1 and 1; row_indices will have rows 1,-1 and 1.

        CE_raw = col_indices * row_indices      # cross edges are -1, rest are 1
        CE_raw = np.triu(CE_raw, k = 1)        # upper triangular part of the matrix to avoid double counting
        CE_raw[CE_raw == 1] = 0             # inverting, to change cross edges from 0 to -1 (and latter to 1)
        cross_edge_arr = (-1)*CE_raw                       # changing cross edges from -1 to 1
        return cross_edge_arr                    # returns upper triangular array of 1 and 0s, 1 for cross edge, 0 for self edge or no edge
    
    def phi_x(self, x, q, adj):                 # returns the angle phi for the basis state/partition x
        """ Returns phi and alpha for the particular basis state/partition x, given the adjacency matrix of the graph."""
        partition_string = self.qubit_sequence(x, q)
        partition_string_copy = partition_string.copy()

        C_E_max_arr_x = self.cross_edges_max_arr(partition_string_copy)
        total_edges = np.sum(adj) / 2  # adj is the adjacency matrix of the given graph
        C_E_actual = C_E_max_arr_x * adj
        C_E_count = np.sum(C_E_actual)
        alpha = -np.pi / (2 * total_edges)
        phi = -2 * alpha * C_E_count
        return phi

    def phases(self, q, adj_matrix):        # runs phi_x for all basis states/partitions; returns the list of PHASES angles, NOT exponential of them, for the whole basis
        N = 2**q
        E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph 
        phase_list = []
        for i in range(N):
            phase_list.append(self.phi_x(i, q, adj_matrix))
        phases = np.array(phase_list)
        return phases

    @staticmethod
    def norm_sq(state):        # returns sum of absolute squares ie. sum of probabilities
        return (np.abs(state)**2).sum()
    
    @staticmethod
    def normalize(state):           # normalizes the state
        if type(state) is not np.ndarray:
            raise ValueError('Error: State is not a numpy array')
        return (1/np.sqrt(QuantumSystem.norm_sq(state)))*state
    
    # @staticmethod
    # def tensor_div(arr,b):            # arr is tenosr prod of two states, b is the second state with all zeroes except one 1 at b'th position
    #     arr_copy = arr.copy()
    # @staticmethod
    # def state_to_qubit:


In [222]:
Q4 = QuantumSystem(4, np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]]))  # 4 qubits, square graph

# Q4.norm_sq(Q4.Identity_N[1:].sum(axis = 0, keepdims = True)*(1/np.sqrt(15)))

# Q4.psi0
# Q4.zero.shape
# Q4.psi0_tensor_zero.shape
# np.where(Q4.Proj_ab != 0)

# Q4.phase_list.shape

# pp(Q4.Swap[0:3, 0:3])
# diff = (Q4.Swap @ Q4.Swap.conj().T)-(np.eye(256))
# np.sum(diff**2)

# # np.where(Q4.Swap @ Q4.Swap.conj().T != 0)

# # np.where(np.eye(256) != 1)[1].shape

#first iter
psi = Q4.psi0_tensor_zero
psi_prime = Q4.U_big @ psi
psi_collapsed_unnorm = Q4.Proj_ab @ psi_prime
psi_collapsed = Q4.normalize(psi_collapsed_unnorm)

#second iter
psi = psi_collapsed
psi_prime = Q4.U_big @ psi
psi_collapsed_unnorm = Q4.Proj_ab @ psi_prime
psi_collapsed = Q4.normalize(psi_collapsed_unnorm)


# #third iter
# psi = psi_collapsed
# psi_prime = Q4.U_big @ psi
# psi_collapsed_unnorm = Q4.Proj_ab @ psi_prime
# psi_collapsed = Q4.normalize(psi_collapsed_unnorm)

# pp(psi.T)
# pp(psi_prime.T)
# pp(psi_collapsed_unnorm.T)
# pp(psi_collapsed_unnorm.conj().T @ psi_collapsed_unnorm)
# pp(psi_collapsed.T)
# pp(psi_collapsed.conj().T @ psi_collapsed)

# print(Q4.p1)
# print(Q4.p2)


In [240]:

f = Q4.psi_iter2.T

f_num = psi_collapsed.T

pp(f_num[0,160])

print(format(161, '8b'))

# pp(Q4.phase_angles.T)

(0.447213595499958-5.476786982642027e-17j)
10100001


In [242]:
#trying for q = 6, line graph

q = 6
adj = np.array([[0, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0], [0, 1, 0, 1, 0, 0], [0, 0, 1, 0, 1, 0], [0, 0, 0, 1, 0, 1], [0, 0, 0, 0, 1, 0]])

Q6 = QuantumSystem(q, adj)


In [277]:
ff = Q6.psi_iter2.T

ff_w= np.where(abs(ff) > .18)

ff_w[1]

partitions = np.array([format(i, '012b') for i in ff_w[1]])

partitions_a = np.array([ i[0:6] for i in partitions])
partitions_b = np.array([ i[6:12] for i in partitions])

ff_w[1] ,partitions_a, (ff[0,2688])



(array([ 640, 1152, 1280, 1344, 1408, 1664, 2368, 2624, 2688, 2752, 2880, 3392]),
 array(['001010', '010010', '010100', '010101', '010110', '011010', '100101', '101001', '101010', '101011', '101101', '110101'], dtype='<U6'),
 (0.21704561184044532-2.6580421380938023e-17j))

In [278]:
#similar for q = 8, line graph

q = 8
adj = np.array([[0, 1, 0, 0, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0], [0, 1, 0, 1, 0, 0, 0, 0], [0, 0, 1, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 1, 0, 0], [0, 0, 0, 0, 1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 1], [0, 0, 0, 0, 0, 0, 1, 0]])

Q8 = QuantumSystem(q, adj)


In [284]:
ff = Q8.psi_iter2.T

ff_w = np.where(abs(ff) > .11)
print(ff_w[1])                          # indices with high amplitude

partitions = np.array([format(i, '016b') for i in ff_w[1]])
partitions_a = np.array([ i[0:8] for i in partitions])

partitions_a, ff[0, 21760], ff[0, 43520]                    # wroking for q = 8, line graph

[21760 43520]


(array(['01010101', '10101010'], dtype='<U8'),
 (0.11140113895957736-1.3642704824821592e-17j),
 (0.11140113895957736-1.3642704824821592e-17j))

In [285]:
# similar for q = 10, line graph

q = 10
adj10 = np.array([ [0,1,0,0,0,0,0,0,0,0], [1,0,1,0,0,0,0,0,0,0], [0,1,0,1,0,0,0,0,0,0], [0,0,1,0,1,0,0,0,0,0], [0,0,0,1,0,1,0,0,0,0], [0,0,0,0,1,0,1,0,0,0], [0,0,0,0,0,1,0,1,0,0], [0,0,0,0,0,0,1,0,1,0], [0,0,0,0,0,0,0,1,0,1], [0,0,0,0,0,0,0,0,1,0] ])
Q10 = QuantumSystem(q, adj10)

In [342]:
ff = Q10.psi_iter2.T
ff_w = np.where(abs(ff) > .0550063)
print(ff_w[1])                          # indices with high amplitude

partitions = np.array([format(i, '020b') for i in ff_w[1]])
partitions_a = np.array([ i[0:10] for i in partitions])

partitions_a, ff[0,ff_w[1]]     # working for q = 10, line graph    

[349184 698368]


(array(['0101010101', '1010101010'], dtype='<U10'),
 array([0.05671645-6.94576181e-18j, 0.05671645-6.94576181e-18j]))

In [343]:
# similar for q = 13, line graph
q = 13
adj13 = np.array([[0,1,0,0,0,0,0,0,0,0,0,0,0], [1,0,1,0,0,0,0,0,0,0,0,0,0], [0,1,0,1,0,0,0,0,0,0,0,0,0], [0,0,1,0,1,0,0,0,0,0,0,0,0], [0,0,0,1,0,1,0,0,0,0,0,0,0], [0,0,0,0,1,0,1,0,0,0,0,0,0], [0,0,0,0,0,1,0,1,0,0,0,0,0], [0,0,0,0,0,0,1,0,1,0,0,0,0], [0,0,0,0,0,0,0,1,0,1,0,0,0], [0,0,0,0,0,0,0,0,1,0,1,0,0], [0,0,0,0,0,0,0,0,0,1,0,1,0], [0,0,0,0,0,0,0,0,0,0,1,0,1], [0,0,0,0,0,0,0,0,0,0,0,0,1]])
Q13 = QuantumSystem(q, adj13)

In [354]:
ff = Q13.psi_iter2.T
ff_w = np.where(abs(ff) > .0201)
print(ff_w[1])                          # indices with high amplitude

partitions = np.array([format(i, '026b') for i in ff_w[1]])
partitions_a = np.array([ i[0:13] for i in partitions])

partitions_a, ff[0,ff_w[1]]     # working for q = 13, line graph

[22364160 44736512]


(array(['0101010101010', '1010101010101'], dtype='<U13'),
 array([0.02042731-2.50162453e-18j, 0.02042731-2.50162453e-18j]))

In [2]:
# similar for q = 16, line graph

q = 14

def adj_line(q):
    adj = np.zeros((q,q))
    for i in range(q):
        if i == 0:
            adj[i, i+1] = 1
        elif i == q-1:
            adj[i, i-1] = 1
        else:
            adj[i, i+1] = 1
            adj[i, i-1] = 1
    return adj

adj14 = adj_line(14)

Q14 = QuantumSystem(q, adj14)

In [18]:
ff = Q14.psi_iter2.T
ff_w = np.where(abs(ff) > .0141)
print(ff_w[1])                          # indices with high amplitude

partitions = np.array([format(i, '028b') for i in ff_w[1]])
partitions_a = np.array([ i[0:14] for i in partitions])

partitions_a, ff[0,ff_w[1]]     # working for q = 14, line graph

[ 44728320  78282752  86671360  88768512  89292800  89423872  89456640
  89473024  89489408  89554944  89817088  90865664  95059968 111837184
 156581888 173359104 177553408 178601984 178864128 178929664 178946048
 178962432 178995200 179126272 179650560 181747712 190136320 223690752]


(array(['00101010101010', '01001010101010', '01010010101010',
        '01010100101010', '01010101001010', '01010101010010',
        '01010101010100', '01010101010101', '01010101010110',
        '01010101011010', '01010101101010', '01010110101010',
        '01011010101010', '01101010101010', '10010101010101',
        '10100101010101', '10101001010101', '10101010010101',
        '10101010100101', '10101010101001', '10101010101010',
        '10101010101011', '10101010101101', '10101010110101',
        '10101011010101', '10101101010101', '10110101010101',
        '11010101010101'], dtype='<U14'),
 array([0.01388594-3.42257633e-03j, 0.01388594-3.42257633e-03j,
        0.01388594-3.42257633e-03j, 0.01388594-3.42257633e-03j,
        0.01388594-3.42257633e-03j, 0.01388594-3.42257633e-03j,
        0.01388594-3.42257633e-03j, 0.01451237-1.77725220e-18j,
        0.01388594-3.42257633e-03j, 0.01388594-3.42257633e-03j,
        0.01388594-3.42257633e-03j, 0.01388594-3.42257633e-03j,
        0.013885

In [19]:
# similar for q = 15, line graph

q = 15
adj15 = adj_line(15)
Q15 = QuantumSystem(q, adj15)


In [21]:
ff = Q15.psi_iter2.T
ff_w = np.where(abs(ff) > .01)
print(ff_w[1])                          # indices with high amplitude

partitions = np.array([format(i, '030b') for i in ff_w[1]])
partitions_a = np.array([ i[0:15] for i in partitions])

partitions_a, ff[0,ff_w[1]]     # working for q = 15, line graph

[178946048 313163776 346718208 355106816 357203968 357728256 357859328
 357892096 357924864 357990400 358252544 359301120 363495424 380272640
 447381504 626327552 693436416 710213632 714407936 715456512 715718656
 715784192 715816960 715849728 715980800 716505088 718602240 726990848
 760545280 894763008]


(array(['001010101010101', '010010101010101', '010100101010101',
        '010101001010101', '010101010010101', '010101010100101',
        '010101010101001', '010101010101010', '010101010101011',
        '010101010101101', '010101010110101', '010101011010101',
        '010101101010101', '010110101010101', '011010101010101',
        '100101010101010', '101001010101010', '101010010101010',
        '101010100101010', '101010101001010', '101010101010010',
        '101010101010100', '101010101010101', '101010101010110',
        '101010101011010', '101010101101010', '101010110101010',
        '101011010101010', '101101010101010', '110101010101010'],
       dtype='<U15'),
 array([0.00992039-2.26426354e-03j, 0.00992039-2.26426354e-03j,
        0.00992039-2.26426354e-03j, 0.00992039-2.26426354e-03j,
        0.00992039-2.26426354e-03j, 0.00992039-2.26426354e-03j,
        0.00992039-2.26426354e-03j, 0.01030469-1.26196032e-18j,
        0.00992039-2.26426354e-03j, 0.00992039-2.26426354e-03j,
       

In [23]:
q = 16
adj16 = adj_line(16)
Q16 = QuantumSystem(q, adj16)

In [24]:
ff = Q16.psi_iter2.T
ff_w = np.where(abs(ff) > .007)
print(ff_w[1])                          # indices with high amplitude

partitions = np.array([format(i, '032b') for i in ff_w[1]])
partitions_a = np.array([ i[0:16] for i in partitions])

partitions_a, ff[0,ff_w[1]]     # working for q = 16, line graph

: 

In [39]:
# create a function to iterate as before, to ultimately get bit strings with the highest probabilities

def iterate_system(q, adj_matrix, iterations = 3):
    start_time = time.time()
    
    Q = QuantumSystem(q, adj_matrix)
    psi = Q.psi0_tensor_zero
    U_big = Q.U_big
    prob_list, psi_iteration_list, psi_prime_list = [], [], []
    prob_psi = []
    unnormalized_psi_list = []

    for i in range(iterations):
        psi_prime = U_big @ psi  # operated state
        psi_prime_list.append(psi_prime)

        psi_collapsed_unnorm = Q.Proj_ab @ psi_prime
        
        unnormalized_psi_list.append(psi_collapsed_unnorm)

        prob = Q.norm_sq(psi_collapsed_unnorm)
        prob_list.append(prob)
        psi = Q.normalize(psi_collapsed_unnorm)
        prob_psi.append(Q.norm_sq(psi))
        

        psi_iteration_list.append(psi)
        
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f'Elapsed time for q = {q}:', elapsed_time)
    return prob_list, psi,  psi_iteration_list, unnormalized_psi_list, Q.psi0_tensor_zero, prob_psi

In [59]:
#testing if fixed the error
q, adj = 3, np.array([[0, 1, 1], [1, 0, 1], [1, 1, 0]])

Q = QuantumSystem(q, adj)
# Q.Phase

it = iterate_system(q, adj, iterations = 10)

Elapsed time for q = 3: 0.025936126708984375


In [67]:
def mask(arr, threshold = .05):
    mask = abs(arr) > threshold
    masked_arr = np.ma.masked_array((arr), mask = ~mask)
    return masked_arr


pp(mask(it[2][0]).T[0])
pp('\n')
pp(mask(it[2][1]).T[0])
pp('\n')
pp(mask(it[2][2]).T[0])
pp('\n')
pp(mask(it[2][3]).T[0])
pp('\n')
pp(mask(it[2][4]).T[0])
pp('\n')
pp(mask(it[2][5]).T[0])
pp('\n')
pp(mask(it[2][6]).T[0])
pp('\n')
pp(mask(it[2][7]).T[0])
pp('\n')
pp('\n')
pp(mask(it[2][8]).T[0])

[-- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j) -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j)
 -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j) -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j) --
 -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.3535533905932738+2.1648901405887332e-17j) -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j) --
 -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j) -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j) -- --
 -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j) -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.3535533905932738+2.1648901405887332e-17j) -- --
 -- -- -- -- -- -- -- -- -- -- -- -- -- (-0.17677669529663684+0.1767766952966369j) -- -- -- -- -- --

IndexError: list index out of range

In [88]:
q , adj = 4, np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]])  # square graph

it = iterate_system(q, adj, iterations = 10)



Elapsed time for q = 4: 0.10970211029052734


In [65]:
it[1].T

array([[ 0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,
         0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,
         0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,
         0.        +0.00000000e+00j,  0.08111071-8.11107106e-02j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,
         0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,
         0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.        +0.00000000e+00j,
         0.        +0.00000000e+00j,  0.        +0.00000000e+00j,  0.08111071-8.11

### nth analytical solution, pushing efficiency to q >23 qubits

In [ ]:
# analytical, extending to nth iteration
import numpy as np
class QuantumSystem:
    def __init__(self, q, adj_matrix):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        
        self.adj_matrix = adj_matrix
        self.E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph
        self.alpha = -np.pi/(2*self.E_t)
        
        #Cached variables
        self.cached_basis_states = None
        self.cached_psi0 = None
        self.cached_zero = None
        self.cached_psi0_tensor_zero = None
        self.cached_I_a =  None
        self.cached_Proj_ab = None
        self.cached_phase_list = None
        # self.cached_U_big = None
        
        self.cached_probs = {0:1 }


        ###########
        self.cached_AbsSq_phase_minus_1 = self.abs_phase_minus_1()
        self.cached_probs = {0:1}                                       # list of probs for mth iter (not prod of probs)
        ###########
        self.cached_prob1 = None
        self.cached_prob2 = None
        self.cached_psi_iter2 = None
        

    # instead of defining a method for creating matrix basis we will do it as a class variable
    @property
    def Identity_N(self):
        if self.cached_basis_states is None:
            self.cached_basis_states = np.eye(self.N)
        return self.cached_basis_states         # returns identity matrix of size 2^q = N
        
    @property
    def psi0(self):
        if self.cached_psi0 is None:
            self.cached_psi0 = (1/np.sqrt(self.N-1))*np.sum(self.Identity_N[1:], axis = 0, keepdims = True).T
        return self.cached_psi0
    
    @property
    def zero(self):
        if self.cached_zero is None:
            self.cached_zero = np.eye(self.N)[0].reshape(1, self.N).T
        return self.cached_zero

    @property
    def psi0_tensor_zero(self):
        if self.cached_psi0_tensor_zero is None:
            self.cached_psi0_tensor_zero = np.kron(self.psi0, self.zero)
        return self.cached_psi0_tensor_zero

    @property
    def Proj_ab(self):
        if self.cached_Proj_ab is None:
            self.cached_Proj_ab = np.kron(self.Identity_N, self.zero @ self.zero.conj().T)  # projection operator
        return self.cached_Proj_ab

    @property
    def phase_angles(self):
        if self.cached_phase_list is None:
            self.cached_phase_list = self.phases(self.q, self.adj_matrix).reshape(self.N, 1)
        return self.cached_phase_list           # COLUMN of phases ANGLES

    @property
    def phase_list(self):       # column of phases (not angles)
        return np.exp(1j*self.phase_angles)
    
    @property
    def Phase(self):        # phase operator
        return np.diag(np.exp(1j*self.phase_angles.reshape(-1)))

###XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX####

    @property
    def p1(self):
        if self.cached_prob1 is None:
            self.cached_prob1 = (.25/(self.N-1))*((np.abs(self.phase_list[1:] - 1)**2).sum())
        return self.cached_prob1

    @property
    def p2(self):
        if self.cached_prob2 is None:
            self.cached_prob2 = ((1/16)/((self.N-1)*self.p1))*((np.abs(self.phase_list[1:] - 1)**4).sum())
        return self.cached_prob2
###XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX####
    #old function    
    # def prob_m(self, m):
    #     def prod_prob_m(m):
    #         if not m:
    #             return 1
    #         p_minus_1 = prod_prob_m(m-1)
    #         p = (1/((self.N-1)*(4**m)*p_minus_1))* np.sum(((self.cached_AbsSq_phase_minus_1)**m))
    #         return p*p_minus_1
        
    #     p_m = (1/((self.N-1)*(4**m)*prod_prob_m(m-1)))* np.sum(((self.cached_AbsSq_phase_minus_1)**m))
    #     return p_m


    def prob_m(self, m):
        if not m:
            return 1

        def prod_prob_m(m):
            if m == 0:
                return 1
            if m in self.cached_probs:              # using cached values recursively, if they exist in the dictionary
                return self.cached_probs[m]*prod_prob_m(m-1)
            
            p_minus_1 = prod_prob_m(m-1) 
            p = (1/((self.N-1)*(4**m)*p_minus_1))* np.sum(((self.cached_AbsSq_phase_minus_1)**m))   # mth prob
            self.cached_probs[m] = p                                            # caching the value of mth prob (not product prob)
            
            return p*p_minus_1

        prod_prob_m(m)
        p_m = self.cached_probs[m]                                              # 
        return p_m
    

    def abs_phase_minus_1(self):           # calc list from ONE to N of abs sq of e^(i*phi) -1. Done for efficiency 
        phases_from1 = np.array(self.phase_list.reshape(-1)[1:])
        phases_diff_abs_sq_from1 = np.abs(phases_from1-1)**2             # (e**i*phi -1)**2 for all except the first elt. len = 2**q -1
        return phases_diff_abs_sq_from1

    #finding d_m
    # def d_m(self, m):
    #     angles = self.phase_angles.T.reshape(-1)[1:]     # row/flattened phase angles starting from ONE
    #     one_minus_cos_to_m = (1- np.cos(angles))**m
    #     sum_one_minus_cos_to_m = np.sum(one_minus_cos_to_m)
    #     d = np.sqrt(self.N-1)/(2**(m/2) * np.sqrt(sum_one_minus_cos_to_m))
    #     return d


    #################################


    # @property
    # def psi_iter2(self):
    #     if self.cached_psi_iter2 is None:
    #         norm_coeff = 1/np.sqrt(4*(self.N -1)*self.p1*self.p2)
    #         psi_iter_2_a = norm_coeff*(((((self.phase_list[1:] - 1))**2)*self.Identity_N[1:]).sum(axis = 0, keepdims = True)).T
    #         self.cached_psi_iter2 = np.kron(psi_iter_2_a, self.zero)                        # projection/collapse on ancilla's zero state
    #     return self.cached_psi_iter2
    
    # @property
    # def psi_iter2_trial(self):
    #     norm_coeff = 1/np.sqrt(4*(self.N -1)*self.p1*self.p2)
    #     psi_iter_2_a = norm_coeff*(((((self.phase_list[1:] - 1))**2)*self.Identity_N[1:]).sum(axis = 0, keepdims = True)).T
        
    #     print('p1:', self.p1)
    #     print('p2:', self.p2)
    #     print(' phase part..:' , (((((self.phase_list[1:] - 1))**2)*self.Identity_N[1:]).sum(axis = 0, keepdims = True)).T)

    #     return psi_iter_2_a

    def psi_iter_m(self,m):
        if m not in self.cached_probs:
            self.prob_m(m)
        
        prod_probs_m = 1
        for i in range(m+1):                    # product of probs from 0 to m (m+1 not included)
            prod_probs_m *= self.cached_probs[i]

        denominator = (2**m)* np.sqrt((self.N-1)* prod_probs_m)

        phases = self.phase_list.reshape(-1)        # phases (not angles). Including the first one coz it is zero anyways, but will affect binary representation
        sum_qm_states = np.power(phases-1, m)        #quantum state except the prefactor

        # print('prod_probs_m:', prod_probs_m)
        # print('denominator:', denominator)
        # print('phases:', phases)
        # print('sum_qm_states:', sum_qm_states)

        return (1/denominator)*sum_qm_states    


    @property
    def Swap(self):
        return self.U_ab()  
    
    @property
    def U_big(self):
        if self.cached_U_big is None:
            self.cached_U_big = self.Swap @ (np.kron(self.Phase, self.Identity_N)) @ self.Swap
        return self.cached_U_big

    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U

    # defining the phase operator after defining the phi_x function for an phase of an individual permutation/basis state

    @staticmethod
    def qubit_sequence(x, q):
        b = np.binary_repr(x, width = q)       # binary string for the integer/basis state x (out of 2**q states) 
        c = np.array([int(i) for i in b])       # eg. '101' -> [1, 0, 1]
        return c
    # permut_arr is the array for the basis state x (which is a permutation of q qubits)
    @staticmethod
    def cross_edges_max_arr(permut_arr):             #all possible cross edges for this permutation (will add graph adjacency matrix later)
        permut_arr[permut_arr == 0] = -1            # calling the zeroes as -1, for distinguishing the cross edges upon multiplication
        col_indices, row_indices = np.meshgrid(permut_arr, permut_arr)  # eg. for permutation 101, col_indices will have columns 1,-1 and 1; row_indices will have rows 1,-1 and 1.

        CE_raw = col_indices * row_indices      # cross edges are -1, rest are 1
        CE_raw = np.triu(CE_raw, k = 1)        # upper triangular part of the matrix to avoid double counting
        CE_raw[CE_raw == 1] = 0             # inverting, to change cross edges from 0 to -1 (and latter to 1)
        cross_edge_arr = (-1)*CE_raw                       # changing cross edges from -1 to 1
        return cross_edge_arr                    # returns upper triangular array of 1 and 0s, 1 for cross edge, 0 for self edge or no edge
    
    def phi_x(self, x, q, adj):                 # returns the angle phi for the basis state/partition x
        """ Returns phi and alpha for the particular basis state/partition x, given the adjacency matrix of the graph."""
        partition_string = self.qubit_sequence(x, q)
        partition_string_copy = partition_string.copy()

        C_E_max_arr_x = self.cross_edges_max_arr(partition_string_copy)
        total_edges = np.sum(adj) / 2  # adj is the adjacency matrix of the given graph
        C_E_actual = C_E_max_arr_x * adj
        C_E_count = np.sum(C_E_actual)
        alpha = -np.pi / (2 * total_edges)
        phi = -2 * alpha * C_E_count
        return phi

    def phases(self, q, adj_matrix):        # runs phi_x for all basis states/partitions; returns the list of PHASES angles, NOT exponential of them, for the whole basis
        N = 2**q
        E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph 
        phase_list = []
        for i in range(N):
            phase_list.append(self.phi_x(i, q, adj_matrix))
        phases = np.array(phase_list)
        return phases

    @staticmethod
    def norm_sq(state):        # returns sum of absolute squares ie. sum of probabilities
        return (np.abs(state)**2).sum()
    
    @staticmethod
    def normalize(state):           # normalizes the state
        if type(state) is not np.ndarray:
            raise ValueError('Error: State is not a numpy array')
        return (1/np.sqrt(QuantumSystem.norm_sq(state)))*state
    
    @staticmethod
    def sorted_binaries(arr, q = None):
        if q is None:
            q = int(np.log2(len(arr)))
        
        if q %1 != 0:
            raise ValueError('Error..The length of the array is not a power of 2...provide argument q manually')
        
        if q < 5:
            cut_off = 2**q
        else:
            cut_off = 2**(q//3)
        
        if not q:
            cut_off = None

        l = np.array(arr).reshape(-1)             # arr is the amplified state (list/np array)
        l_abs_sq = np.abs(l)**2

        en_l = sorted(list(enumerate(l_abs_sq)), reverse = True, key = lambda x: x[1])[:cut_off]   #keeping on the best partitions

        en_binaries = [(format(i[0], '>0'+str(q)+'b'), i[1]) for i in en_l]

        return en_binaries
      

### Sparse matrices rough work


In [ ]:
# creating full matrix using sparse
import numpy as np


(4, 4)


array([[0., 1., 0., 1.],
       [1., 0., 1., 0.],
       [0., 1., 0., 1.],
       [1., 0., 1., 0.]])

In [ ]:
# doing a toy example using sparse matrices
q = 3
adj = np.array([[0, 1, 1], [1, 0, 1], [1, 1, 0]])

#manually writing the sparse matrices

from scipy.sparse import csr_matrix as csr

I3 = csr(np.eye(3))


[ 4. 18. 25.  2.  5.]
